# Month 3 — Part 4: Regression Error Analysis

## Goal

Move beyond aggregate RMSE and identify where the tuned Ridge model makes its largest errors.

We will use out-of-fold predictions so that every residual is computed from a prediction made by a model that did not train on that row.

For each row:

$$
r_i = y_i - \hat{y}_i
$$

and

$$
|r_i| = |y_i - \hat{y}_i|
$$

We will analyze residuals by:

- `session_seen`
- language
- `history_seen`
- user activity
- worst individual residuals

The final holdout set remains untouched.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

ROOT = Path.cwd().parents[1]

df = pd.read_csv(ROOT / "data" / "duolingo_flagship_v5.csv")
split_df = pd.read_csv(ROOT / "data" / "split_users.csv")

cv_users = set(split_df.loc[split_df["split"] == "cv", "user_id"])
df_cv = df[df["user_id"].isin(cv_users)].copy()

features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = df_cv[features]
y = df_cv["p_recall"]
groups = df_cv["user_id"]

gkf = GroupKFold(n_splits=5)

ridge_pipe = Pipeline([("scaler", StandardScaler()),("ridge", Ridge(alpha=30))])

oof_pred = np.empty(len(df_cv))

for train_idx, val_idx in gkf.split(X, y, groups):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]

    ridge_pipe.fit(X_train, y_train)
    oof_pred[val_idx] = ridge_pipe.predict(X_val)

df_cv["oof_pred"] = oof_pred
df_cv["residual"] = df_cv["p_recall"] - df_cv["oof_pred"]
df_cv["abs_residual"] = df_cv["residual"].abs()

rmse = np.sqrt(mean_squared_error(df_cv["p_recall"], df_cv["oof_pred"]))

print("OOF RMSE:", rmse)
print("NaN predictions:", df_cv["oof_pred"].isna().sum())
print(df_cv[["p_recall", "oof_pred", "residual", "abs_residual"]].head())

OOF RMSE: 0.2736617069127015
NaN predictions: 0
   p_recall  oof_pred  residual  abs_residual
0       1.0  0.873585  0.126415      0.126415
1       1.0  0.903194  0.096806      0.096806
2       1.0  0.907277  0.092723      0.092723
3       1.0  0.912463  0.087537      0.087537
4       1.0  0.869357  0.130643      0.130643


In [2]:
session_counts = (df_cv["session_seen"].value_counts().sort_index())

print(session_counts.head(20))
print("\nMax session_seen:", df_cv["session_seen"].max())

session_seen
1     8623
2     2844
3     1489
4      731
5      394
6      174
7       84
8       40
9       20
10       9
11      11
12       8
13       3
14       4
15       1
16       3
Name: count, dtype: int64

Max session_seen: 16


In [3]:
df_cv["session_seen_bucket"] = pd.cut(
    df_cv["session_seen"],
    bins=[0, 1, 2, 3, 4, np.inf],
    labels=["1", "2", "3", "4", "5+"]
)

In [4]:
session_analysis = (
    df_cv.groupby("session_seen_bucket", observed=True)
    .agg(
        count=("p_recall", "size"),
        target_mean=("p_recall", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean")
    )
)

session_analysis["rmse"] = (
    df_cv.groupby("session_seen_bucket", observed=True)["residual"]
    .apply(lambda x: np.sqrt(np.mean(x ** 2)))
)

print(session_analysis)

                     count  target_mean  mean_residual       mae      rmse
session_seen_bucket                                                       
1                     8623     0.878928      -0.016001  0.198011  0.324039
2                     2844     0.914557       0.019373  0.154468  0.201504
3                     1489     0.919409       0.024855  0.135007  0.156052
4                      731     0.922025       0.029936  0.121993  0.136506
5+                     751     0.916248       0.028680  0.101771  0.118176


### Session-size finding

Prediction error decreases sharply as `session_seen` increases.

Rows with `session_seen = 1` have an OOF RMSE of about `0.324`, compared with about `0.118` for `session_seen >= 5`.

Although `session_seen = 1` rows make up roughly 60% of the CV pool, they account for about 84% of total squared error.

This strongly supports the hypothesis that small-denominator target noise is a major driver of the flagship model's remaining error.

In [5]:
print(df_cv.columns.tolist())

['practice_time', 'user_id', 'ui_language', 'learning_language', 'surface_form', 'lemma', 'pos', 'grammar_tags', 'lag_days', 'history_seen', 'history_correct', 'session_seen', 'session_correct', 'p_recall', 'lexeme_id', 'lag_days_log', 'history_accuracy', 'difficulty_rank_in_language', 'oof_pred', 'residual', 'abs_residual', 'session_seen_bucket']


In [6]:
print("Learning languages:")
print(df_cv["learning_language"].value_counts())

print("\nUI languages:")
print(df_cv["ui_language"].value_counts())

Learning languages:
learning_language
en    5926
es    3829
fr    1732
de    1551
it     939
pt     461
Name: count, dtype: int64

UI languages:
ui_language
en    8512
es    4358
pt    1002
it     566
Name: count, dtype: int64


In [7]:
language_analysis = (
    df_cv.groupby("learning_language")
    .agg(
        count=("p_recall", "size"),
        target_mean=("p_recall", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean")
    )
)

language_analysis["rmse"] = (
    df_cv.groupby("learning_language")["residual"]
    .apply(lambda x: np.sqrt(np.mean(x ** 2)))
)

language_analysis = language_analysis.sort_values("rmse", ascending=False)

print(language_analysis)

                   count  target_mean  mean_residual       mae      rmse
learning_language                                                       
pt                   461     0.793275      -0.096406  0.261530  0.401317
de                  1551     0.888034      -0.006670  0.178986  0.285503
fr                  1732     0.883701      -0.007835  0.180888  0.280318
it                   939     0.899288       0.001900  0.168703  0.270172
en                  5926     0.898226       0.002459  0.169342  0.264923
es                  3829     0.906287       0.012942  0.167142  0.260608


In [8]:
language_session = (
    df_cv.groupby("learning_language")
    .agg(
        count=("session_seen", "size"),
        mean_session_seen=("session_seen", "mean"),
        median_session_seen=("session_seen", "median"),
        session_seen_1_ratio=("session_seen", lambda x: (x == 1).mean())
    )
)

print(language_session.sort_values("session_seen_1_ratio", ascending=False))

                   count  mean_session_seen  median_session_seen  \
learning_language                                                  
pt                   461           1.626898                  1.0   
de                  1551           1.835590                  1.0   
it                   939           1.837061                  1.0   
es                  3829           1.770697                  1.0   
en                  5926           1.807965                  1.0   
fr                  1732           1.993649                  1.0   

                   session_seen_1_ratio  
learning_language                        
pt                             0.713666  
de                             0.626692  
it                             0.625133  
es                             0.610081  
en                             0.584036  
fr                             0.541570  


In [9]:
language_by_session = (
    df_cv.groupby(
        ["session_seen_bucket", "learning_language"],
        observed=True
    )
    .agg(
        count=("p_recall", "size"),
        target_mean=("p_recall", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean")
    )
)

language_by_session["rmse"] = (
    df_cv.groupby(
        ["session_seen_bucket", "learning_language"],
        observed=True
    )["residual"]
    .apply(lambda x: np.sqrt(np.mean(x ** 2)))
)

print(language_by_session.loc["1"].sort_values("rmse", ascending=False))

                   count  target_mean  mean_residual       mae      rmse
learning_language                                                       
pt                   329     0.738602      -0.148505  0.312065  0.461918
fr                   938     0.864606      -0.026830  0.211508  0.340342
de                   972     0.870370      -0.025735  0.202444  0.332395
it                   587     0.885860      -0.014194  0.189150  0.316372
en                  3461     0.887027      -0.008630  0.191239  0.314096
es                  2336     0.894264      -0.000316  0.186945  0.306171


### Language finding

Portuguese (`pt`) is a clear high-error subgroup.

Overall, Portuguese rows have an OOF RMSE of about `0.401` and a mean residual of about `-0.096`, indicating systematic overprediction.

Portuguese also has a larger share of `session_seen = 1` rows, so part of its higher RMSE is explained by greater target noise.

However, after restricting the comparison to `session_seen = 1`, Portuguese remains substantially worse:

- Portuguese RMSE: `0.462`
- Portuguese mean residual: `-0.149`
- Portuguese target mean: `0.739`

The other languages have much smaller systematic residuals in the same session-size group.

This suggests that denominator noise alone does not explain the Portuguese error. The current feature set does not include `learning_language`, so language-specific or correlated cohort effects may represent missing predictive signal. This is an error-analysis finding, not yet a causal conclusion.

In [10]:
print(df_cv["history_seen"].describe())

print("\nQuantiles:")
print(
    df_cv["history_seen"].quantile(
        [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    )
)

count    14438.000000
mean        16.421665
std         52.265991
min          1.000000
25%          3.000000
50%          6.000000
75%         14.000000
max       2202.000000
Name: history_seen, dtype: float64

Quantiles:
0.00       1.0
0.25       3.0
0.50       6.0
0.75      14.0
0.90      32.0
0.95      55.0
0.99     147.0
1.00    2202.0
Name: history_seen, dtype: float64


In [11]:
df_cv["history_seen_bucket"] = pd.cut(
    df_cv["history_seen"],
    bins=[0, 3, 6, 14, 32, np.inf],
    labels=["1-3", "4-6", "7-14", "15-32", "33+"]
)

history_analysis = (
    df_cv.groupby("history_seen_bucket", observed=True)
    .agg(
        count=("p_recall", "size"),
        target_mean=("p_recall", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        session_seen_1_ratio=("session_seen", lambda x: (x == 1).mean())
    )
)

history_analysis["rmse"] = (
    df_cv.groupby("history_seen_bucket", observed=True)["residual"]
    .apply(lambda x: np.sqrt(np.mean(x ** 2)))
)

print(history_analysis)

                     count  target_mean  mean_residual       mae  \
history_seen_bucket                                                
1-3                   3806     0.886546      -0.009093  0.180251   
4-6                   3460     0.889786      -0.001993  0.177675   
7-14                  3598     0.901165       0.006630  0.167864   
15-32                 2146     0.901626       0.005090  0.168605   
33+                   1428     0.897038       0.002996  0.172828   

                     session_seen_1_ratio      rmse  
history_seen_bucket                                  
1-3                              0.595113  0.284497  
4-6                              0.598555  0.278503  
7-14                             0.600056  0.263134  
15-32                            0.615098  0.266345  
33+                              0.565826  0.269208  


### History-depth finding

Error varies only modestly across `history_seen` buckets.

RMSE decreases from about `0.284` for low-history rows to about `0.263` around `7-14` prior observations, then remains near `0.266-0.269`.

Mean residuals stay close to zero across all buckets, indicating little systematic over- or underprediction.

Unlike `session_seen` and Portuguese language, `history_seen` does not appear to be a major concentrated failure mode for the current Ridge model.

In [12]:
user_activity = df_cv.groupby("user_id").size()

df_cv["user_activity"] = df_cv["user_id"].map(user_activity)

print(df_cv["user_activity"].describe())

print("\nQuantiles:")
print(
    df_cv["user_activity"].quantile(
        [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    )
)

count    14438.000000
mean        23.917717
std         25.519079
min          1.000000
25%          6.000000
50%         15.000000
75%         32.000000
max        130.000000
Name: user_activity, dtype: float64

Quantiles:
0.00      1.0
0.25      6.0
0.50     15.0
0.75     32.0
0.90     60.0
0.95     74.0
0.99    115.0
1.00    130.0
Name: user_activity, dtype: float64


In [13]:
df_cv["user_activity_bucket"] = pd.cut(
    df_cv["user_activity"],
    bins=[0, 6, 15, 32, 60, np.inf],
    labels=["1-6", "7-15", "16-32", "33-60", "61+"]
)

activity_analysis = (
    df_cv.groupby("user_activity_bucket", observed=True)
    .agg(
        count=("p_recall", "size"),
        target_mean=("p_recall", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        session_seen_1_ratio=("session_seen", lambda x: (x == 1).mean())
    )
)

activity_analysis["rmse"] = (
    df_cv.groupby("user_activity_bucket", observed=True)["residual"]
    .apply(lambda x: np.sqrt(np.mean(x ** 2)))
)

print(activity_analysis)

                      count  target_mean  mean_residual       mae  \
user_activity_bucket                                                
1-6                    3890     0.897826       0.008407  0.166985   
7-15                   3486     0.898066       0.003874  0.172484   
16-32                  3506     0.892609      -0.003253  0.177131   
33-60                  2189     0.899114       0.000651  0.171949   
61+                    1367     0.870702      -0.028295  0.193946   

                      session_seen_1_ratio      rmse  
user_activity_bucket                                  
1-6                               0.436761  0.250949  
7-15                              0.613597  0.270377  
16-32                             0.649743  0.282253  
33-60                             0.686615  0.277513  
61+                               0.734455  0.312580  


In [14]:
activity_given_n1 = (
    df_cv[df_cv["session_seen"] == 1]
    .groupby("user_activity_bucket", observed=True)
    .agg(
        count=("p_recall", "size"),
        target_mean=("p_recall", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean")
    )
)

activity_given_n1["rmse"] = (
    df_cv[df_cv["session_seen"] == 1]
    .groupby("user_activity_bucket", observed=True)["residual"]
    .apply(lambda x: np.sqrt(np.mean(x ** 2)))
)

print(activity_given_n1)

                      count  target_mean  mean_residual       mae      rmse
user_activity_bucket                                                       
1-6                    1699     0.879341      -0.010223  0.202501  0.324857
7-15                   2139     0.883590      -0.009960  0.195422  0.318452
16-32                  2278     0.876207      -0.019184  0.199537  0.326910
33-60                  1503     0.890220      -0.008949  0.186439  0.310922
61+                    1004     0.857570      -0.041983  0.209794  0.346379


### User-activity finding

Overall RMSE appears to increase with the number of rows associated with a user, but user activity is strongly confounded with `session_seen`.

Highly active users have a much larger proportion of `session_seen = 1` observations, which are intrinsically noisier.

After restricting the comparison to `session_seen = 1`, most of the apparent activity trend disappears. RMSE remains roughly in the `0.31-0.35` range across activity buckets.

The `61+` group still shows somewhat higher error (`RMSE ≈ 0.346`) and a negative mean residual (`≈ -0.042`), suggesting mild systematic overprediction for the most active-user cohort.

This effect is substantially weaker than the Portuguese-language failure mode and may still be explained by other correlated cohort characteristics.

In [16]:
worst_50 = (df_cv.nlargest(50, "abs_residual"))

cols = [
    "user_id",
    "learning_language",
    "p_recall",
    "oof_pred",
    "residual",
    "abs_residual",
    "session_seen",
    "history_seen",
    "history_accuracy",
    "lag_days"
]

display(worst_50[cols])

,user_id,learning_language,p_recall,oof_pred,residual,abs_residual,session_seen,history_seen,history_accuracy,lag_days
8805,u:fM1O,fr,0.0,0.933767,-0.933767,0.933767,1,103,0.980583,0.002
10954,u:g2-p,en,0.0,0.930621,-0.930621,0.930621,1,32,1.000000,0.004
3119,u:iIUK,en,0.0,0.930326,-0.930326,0.930326,1,21,1.000000,0.009
15489,u:gRv8,en,0.0,0.930268,-0.930268,0.930268,1,17,1.000000,0.003
14714,u:fX4E,de,0.0,0.930245,-0.930245,0.930245,1,15,1.000000,0.001
2372,u:k_k,de,0.0,0.930134,-0.930134,0.930134,1,11,1.000000,0.002
11236,u:fCCY,en,0.0,0.930130,-0.930130,0.930130,1,14,1.000000,0.008
10106,u:ffm3,de,0.0,0.930126,-0.930126,0.930126,1,14,1.000000,0.002
16107,u:gAuM,pt,0.0,0.930061,-0.930061,0.930061,1,8,1.000000,0.002
2434,u:hSSV,fr,0.0,0.930058,-0.930058,0.930058,1,10,1.000000,0.006


In [17]:
print("session_seen = 1 ratio:",(worst_50["session_seen"] == 1).mean())
print("p_recall = 0 ratio:",(worst_50["p_recall"] == 0).mean())
print("\nLanguage counts:")
print(worst_50["learning_language"].value_counts())
print("\nUnique users:",worst_50["user_id"].nunique())

session_seen = 1 ratio: 1.0
p_recall = 0 ratio: 1.0

Language counts:
learning_language
en    16
de     9
es     9
pt     8
fr     7
it     1
Name: count, dtype: int64

Unique users: 40


### Worst-error finding

The 50 largest absolute residuals all come from rows with `session_seen = 1` and `p_recall = 0`.

In these cases, the Ridge model often predicts recall probabilities near `0.93`, typically because the learner has strong previous history and a very short lag, but the single observed Bernoulli trial happens to be incorrect.

This is consistent with the noise-floor analysis: when the target is based on only one trial, even a reasonable probability estimate can produce an extremely large observed residual.

The 50 worst rows span 40 unique users, so this is not driven by a single anomalous learner.

Portuguese is also strongly overrepresented: it accounts for about 16% of the worst 50 rows despite representing only about 3.2% of the CV pool, reinforcing the earlier evidence of a Portuguese-specific or correlated cohort effect.

## Part 4 Summary

Regression error analysis showed that the remaining Ridge error is highly structured.

The strongest driver is `session_seen`. Rows with only one observed trial have much higher RMSE than rows with larger denominators, which strongly supports the binomial-noise interpretation from Part 2.

Portuguese (`pt`) is a clear high-error subgroup. Its error remains substantially larger even after conditioning on `session_seen = 1`, and the model systematically overpredicts recall for this group. This may indicate missing language-specific or correlated cohort signal.

`history_seen` does not appear to be a major concentrated failure mode. Error differences across history-depth buckets are relatively small and mean residuals remain near zero.

Apparent error differences across user-activity groups are largely explained by their different `session_seen` distributions, although the most active-user cohort shows mild systematic overprediction.

Finally, all 50 largest absolute residuals come from `session_seen = 1` rows with `p_recall = 0`. These extreme errors are consistent with rare failures from observations whose underlying recall probability may still be high.

Overall, the analysis suggests that a large fraction of the remaining error is driven by target noise, while Portuguese-language behavior represents the clearest candidate for additional learnable signal.